# Retail Conversion Intelligence — TP Final Integrador

**Seminario de Actualización · IFTS N.º 18 · 2026**  
**Autora:** Fernanda Fava

En este trabajo voy a entrenar un modelo para estimar si una sesión de navegación en una tienda online termina en una compra. Después voy a guardar el modelo y utilizarlo desde una API hecha con FastAPI.

**Fuente:** [Online Shoppers Purchasing Intention Dataset — UCI](https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset), ID 468. Es un proyecto académico, no una herramienta comercial validada.

## 1. Librerías

Importo las librerías que voy a usar para analizar los datos, entrenar, evaluar y guardar el modelo.

In [1]:
from pathlib import Path
import os
import sys
import joblib
import sklearn
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, precision_score, recall_score

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)

Python: 3.12.10
scikit-learn: 1.9.1


## 2. Carga del dataset

Voy a trabajar con los datos de UCI. Si el CSV ya está guardado en `data/`, lo leo desde allí. Si todavía no existe, lo descargo y guardo una copia. La variable objetivo es `Revenue` (compra: sí o no).

In [2]:
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "notebook":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
archivo = DATA_DIR / "online_shoppers_intention.csv"

if archivo.exists():
    df = pd.read_csv(archivo)
    print("CSV local cargado:", archivo)
else:
    from ucimlrepo import fetch_ucirepo
    dataset = fetch_ucirepo(id=468)
    df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
    df.to_csv(archivo, index=False)
    print("Dataset descargado y guardado en:", archivo)

print("Dimensiones:", df.shape)
display(df.head())

CSV local cargado: C:\Users\Noxi-PC\Documents\TP_Final\data\online_shoppers_intention.csv
Dimensiones: (12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


## 3. Análisis exploratorio

Voy a revisar las primeras filas, los valores faltantes, las filas repetidas y la proporción de sesiones que terminaron en compra.

In [3]:
print("VALORES NULOS:", df.isna().sum().sum())
print("REGISTROS DUPLICADOS:", df.duplicated().sum())
print("\nDISTRIBUCIÓN DE COMPRAS:")
print(df["Revenue"].value_counts())
print("\nPORCENTAJE DE COMPRAS:")
print(df["Revenue"].value_counts(normalize=True).mul(100).round(2))

VALORES NULOS: 0
REGISTROS DUPLICADOS: 125

DISTRIBUCIÓN DE COMPRAS:
Revenue
False    10422
True      1908
Name: count, dtype: int64

PORCENTAJE DE COMPRAS:
Revenue
False    84.53
True     15.47
Name: proportion, dtype: float64


### Interpretación inicial

El dataset tiene **12.330 filas y 18 columnas**. No encontré valores nulos, pero sí **125 registros duplicados**. El **84,53 %** de las sesiones no terminó en compra y el **15,47 %** sí.

Las clases están desbalanceadas, por eso no voy a evaluar el modelo solamente con accuracy. También revisaré F1, precision y recall.

Las filas iguales podrían ser sesiones diferentes porque no tenemos un identificador único. Por ahora **no las elimino**; dejo esta decisión documentada.

In [4]:
# Ver algunas de las filas repetidas sin modificar el dataset
repetidas = df[df.duplicated(keep=False)]
print("Filas dentro de grupos repetidos:", len(repetidas))
display(repetidas.head(10))

Filas dentro de grupos repetidos: 201


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
85,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,1,1,1,3,Returning_Visitor,False,False
132,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,3,2,3,3,Returning_Visitor,False,False
158,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,1,1,1,3,Returning_Visitor,False,False
159,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,3,2,3,3,Returning_Visitor,False,False
178,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,3,2,3,3,Returning_Visitor,False,False
252,0,0.0,0,0.0,2,0.0,0.2,0.2,0.0,0.0,Mar,1,1,1,1,Returning_Visitor,False,False
286,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Mar,2,2,1,1,Returning_Visitor,False,False
293,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Mar,1,1,1,1,Returning_Visitor,True,False
298,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Mar,1,1,8,1,Returning_Visitor,False,False
330,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Mar,3,2,3,1,Returning_Visitor,False,False


## 4. Selección de variables

Voy a utilizar ocho variables de navegación. No incluyo `PageValues` porque quiero probar el modelo con datos de comportamiento que puedan estar disponibles durante una sesión. `ProductRelated_Duration` se mide en segundos.

In [5]:
features = [
    "ProductRelated", "ProductRelated_Duration", "BounceRates",
    "ExitRates", "Administrative", "VisitorType", "Weekend", "Month"
]
target = "Revenue"
X = df[features].copy()
y = df[target].astype(int)
print("Variables:", features)
print("Objetivo:", target)
print("Datos de entrada:", X.shape)
print("Valores faltantes en las variables seleccionadas:", X.isna().sum().sum())
print("Meses:", sorted(X["Month"].unique()))
print("Tipos de visitante:", sorted(X["VisitorType"].unique()))

Variables: ['ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'Administrative', 'VisitorType', 'Weekend', 'Month']
Objetivo: Revenue
Datos de entrada: (12330, 8)
Valores faltantes en las variables seleccionadas: 0
Meses: ['Aug', 'Dec', 'Feb', 'Jul', 'June', 'Mar', 'May', 'Nov', 'Oct', 'Sep']
Tipos de visitante: ['New_Visitor', 'Other', 'Returning_Visitor']


## 5. División de datos y entrenamiento

Reservo el 20 % para prueba y uso una división estratificada para mantener la proporción de compras. Comparo regresión logística y Random Forest con validación cruzada sobre entrenamiento. La selección se hace sin mirar el conjunto de prueba.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
numericas = ["ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates", "Administrative"]
categoricas = ["VisitorType", "Weekend", "Month"]

def preparacion():
    return ColumnTransformer([
        ("numericas", StandardScaler(), numericas),
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), categoricas)
    ])

modelos = {
    "Regresión logística": Pipeline([
        ("preparacion", preparacion()),
        ("modelo", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("preparacion", preparacion()),
        ("modelo", RandomForestClassifier(n_estimators=150, min_samples_leaf=3,
                                          class_weight="balanced_subsample", random_state=42, n_jobs=-1))
    ])
}

validacion = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados_cv = {
    nombre: cross_val_score(modelo, X_train, y_train, cv=validacion, scoring="f1").mean()
    for nombre, modelo in modelos.items()
}
display(pd.Series(resultados_cv, name="F1 medio").sort_values(ascending=False))

Random Forest          0.388812
Regresión logística    0.385817
Name: F1 medio, dtype: float64

## 6. Evaluación

Elijo el modelo con mejor F1 en validación cruzada. Después lo entreno con todo el conjunto de entrenamiento y lo evalúo una sola vez en prueba. Voy a mirar también los falsos positivos y falsos negativos.

In [7]:
nombre_ganador = max(resultados_cv, key=resultados_cv.get)
modelo_final = modelos[nombre_ganador].fit(X_train, y_train)
predicciones = modelo_final.predict(X_test)
probabilidades = modelo_final.predict_proba(X_test)[:, 1]

metricas = {
    "modelo": nombre_ganador,
    "f1_test": float(f1_score(y_test, predicciones)),
    "precision_test": float(precision_score(y_test, predicciones, zero_division=0)),
    "recall_test": float(recall_score(y_test, predicciones, zero_division=0)),
    "roc_auc_test": float(roc_auc_score(y_test, probabilidades))
}
print("Modelo elegido:", nombre_ganador)
print("Matriz de confusión:")
print(confusion_matrix(y_test, predicciones))
print(classification_report(y_test, predicciones, zero_division=0))
print(metricas)

Modelo elegido: Random Forest
Matriz de confusión:
[[1840  244]
 [ 246  136]]
              precision    recall  f1-score   support

           0       0.88      0.88      0.88      2084
           1       0.36      0.36      0.36       382

    accuracy                           0.80      2466
   macro avg       0.62      0.62      0.62      2466
weighted avg       0.80      0.80      0.80      2466

{'modelo': 'Random Forest', 'f1_test': 0.3569553805774278, 'precision_test': 0.35789473684210527, 'recall_test': 0.35602094240837695, 'roc_auc_test': 0.7553210197867573}


## 7. Guardado del modelo

Guardo el preprocesamiento y el clasificador juntos en `modelo.joblib`, para que la API pueda recibir las variables originales. También registro la versión de scikit-learn y el tamaño del archivo, como pide la consigna.

In [8]:
ruta_modelo = PROJECT_DIR / "app" / "modelo" / "modelo.joblib"
ruta_modelo.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(modelo_final, ruta_modelo)
print("Modelo guardado en:", ruta_modelo)
print("Versión de scikit-learn:", sklearn.__version__)
print("Tamaño en MB:", round(os.path.getsize(ruta_modelo) / 1e6, 3))

Modelo guardado en: C:\Users\Noxi-PC\Documents\TP_Final\app\modelo\modelo.joblib
Versión de scikit-learn: 1.9.1
Tamaño en MB: 19.071


### Interpretación de resultados

Comparé Random Forest y Regresión Logística. Los dos tuvieron resultados muy parecidos en validación cruzada.
El modelo seleccionado fue Random Forest. En los datos de prueba obtuvo un F1 de 0,357 para la clase compra y un ROC-AUC de 0,755.
El modelo detectó 136 de las 382 compras reales, pero no logró identificar otras 246. Esto muestra que todavía necesita mejoras.
Como hay muchas más sesiones sin compra, la exactitud general no es suficiente para evaluar el resultado. Por eso también analicé precision, recall y F1.

### 6.1. Prueba de mejoras

Como el dataset tiene muchas más sesiones sin compra que con compra, voy a probar otras configuraciones de los modelos.
Voy a comparar los resultados mediante validación cruzada, utilizando solamente los datos de entrenamiento. El objetivo es mejorar la identificación de compras sin utilizar los datos de prueba para elegir el modelo.